In [1]:
import pandas as pd
import duckdb
import os
import os, json
from PyQt5.QtSql import QSqlDatabase, QSqlTableModel
from uuid import uuid4
import pandas as pd
pd.set_option("display.max_colwidth", 200)
#pd.set_option("display.width", 500)  # adjusts total line width before wrapping
import numpy as np
import pickle
#from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize
import gspread
from gspread_dataframe import get_as_dataframe, set_with_dataframe
from google.oauth2 import service_account # based on google-auth library
import matplotlib.pyplot as plt
import re
from sklearn.preprocessing import normalize

from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression

import duckdb
import unicodedata
from transformers import AutoTokenizer, AutoModel
import importlib
from sklearn.manifold import TSNE
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

import h5py  # or np.save

import faiss

from sklearn.metrics import confusion_matrix

import seaborn as sns
import matplotlib.pyplot as plt

import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

#conn = duckdb.connect('/srv/data/grela/grela_v0-3.duckdb', read_only=True)

In [2]:
# our own embedding functions
import helper_functions
importlib.reload(helper_functions)

<module 'helper_functions' from '/home/jupyter-vojta/notebooks/enemy-christianity/scripts/helper_functions.py'>

In [3]:
try:
    file_data = json.load(open(os.path.expanduser("~/ServiceAccountsKey.json")))
    # (2) transform the content into crendentials object
    credentials = service_account.Credentials.from_service_account_info(file_data)
# (3) specify your usage of the credentials
    scoped_credentials = credentials.with_scopes(['https://spreadsheets.google.com/feeds', 'https://www.googleapis.com/auth/drive'])
# (4) use the constrained credentials for authentication of gspread package
    gc = gspread.Client(auth=scoped_credentials)
    enemy_gs = gc.open_by_url("https://docs.google.com/spreadsheets/d/1ENbQuxY_U8W0sKHtiN348AS7KS9pxKjnbSg18KuUs7U/edit?usp=sharing")
except:
    pass

## Embeddings for grouped

In [4]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

tokenizer_agbert = AutoTokenizer.from_pretrained("pranaydeeps/Ancient-Greek-BERT")
model_agbert = AutoModel.from_pretrained("pranaydeeps/Ancient-Greek-BERT")

/srv/venvs/greek_venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [5]:
with open ("/srv/data/enemy-christ/large-data/grouped_df.pkl", "rb") as f:
    grouped = pickle.load(f)

In [6]:
len(grouped)

1001301

In [7]:
# exclude GNT
grouped_sample = grouped[~grouped["sentence_id"].str.startswith("lagt_tlg0031")].sample(10000, random_state=42).copy()

In [8]:
# Compute embeddings
device = "cuda"
emb_matrix = helper_functions.bert_sentence_embeddings(
    list(grouped_sample["sent_text"].apply(lambda text: helper_functions.normalize_greek(text, strip_diacritics=True))),
    tokenizer=tokenizer_agbert,
    model=model_agbert,
    device=device,
    batch_size=32,           # adjust for your GPU/CPU RAM
    max_length=tokenizer_agbert.model_max_length,
    l2_normalize=True,       # set False if you prefer raw vectors
)

grouped_sample["sent_emb_agbert"] = [emb for emb in emb_matrix]

In [9]:
import joblib
best_clf = joblib.load("../data/polemical_bin_mlp_agbert.joblib")

In [10]:
# Apply classifier
prob_threshold = 0.9
X_new = np.stack(grouped_sample["sent_emb_agbert"].values)
proba = best_clf.predict_proba(X_new)[:, 1]  # probability for "polemical" (class=1)
y_pred = (proba >= prob_threshold).astype(int)

# Store results
grouped_sample["polemical_pred"] = y_pred

# Quick summary
n_total = len(grouped_sample)
n_polemical = (y_pred == 1).sum()   # assuming 1 = polemical, 0 = unpolemical
n_unpolemical = (y_pred == 0).sum()

print(f"Total sentences: {n_total}")
print(f"Polemical: {n_polemical} ({n_polemical/n_total:.2%})")
print(f"Unpolemical: {n_unpolemical} ({n_unpolemical/n_total:.2%})")

Total sentences: 10000
Polemical: 922 (9.22%)
Unpolemical: 9078 (90.78%)


In [12]:
texts = (
    helper_functions.normalize_greek(t, strip_diacritics=True)
    for t in grouped["sent_text"].tolist()
)

In [14]:
import os, math, time
import numpy as np
import torch
from tqdm import tqdm

# 1) Make tokenizers behave nicely in Jupyter + avoid extra threads
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# 2) Put model in eval + no grad
model_agbert.eval()

# 3) Choose device (GPU if you have it)
device = "cuda" if torch.cuda.is_available() else "cpu"
model_agbert.to(device)

# 4) Pre-normalize texts (keeps whitespace; no diacritics stripping if you don’t want it)
texts = grouped["sent_text"].apply(
    lambda t: helper_functions.normalize_greek(t, strip_diacritics=True)
).tolist()

N   = len(texts)
dim = model_agbert.config.hidden_size  # usually 768

# Tunables — start conservative on CPU; you can raise after you see stable progress
OUTER = 1024    # sentences per outer chunk
INNER = 128     # batch_size inside the helper
MAXLEN = min(512, tokenizer_agbert.model_max_length)  # cap at 512 to be safe

emb_path = "../data/large-data/greek_sent_embeddings.memmap"
emb_mm   = np.memmap(emb_path, dtype="float32", mode="w+", shape=(N, dim))

# 5) Chunked loop with progress and per-chunk flush
offset = 0
t0 = time.time()

with torch.inference_mode():
    for i in tqdm(range(0, N, OUTER), total=math.ceil(N/OUTER), desc="Embedding"):
        chunk = texts[i:i+OUTER]

        embs = helper_functions.bert_sentence_embeddings(
            chunk,
            tokenizer=tokenizer_agbert,
            model=model_agbert,
            device=device,             # "cuda" if available
            batch_size=INNER,          # inner micro-batch
            max_length=MAXLEN,         # cap at 512
            l2_normalize=True,
        )                              # shape: [len(chunk), dim]

        emb_mm[offset:offset+len(chunk)] = embs
        offset += len(chunk)

        # flush to disk every few chunks (so you see IO activity & safeguard against crashes)
        if (i // OUTER) % 10 == 0:
            emb_mm.flush()

emb_mm.flush()
del emb_mm

print(f"Done. {N} sentences → {dim}-d vectors in {time.time()-t0:.1f}s")

Embedding: 100%|██████████| 978/978 [1:07:41<00:00,  4.15s/it]

Done. 1001301 sentences → 768-d vectors in 4061.2s


In [15]:

# open your memmap
emb_read = np.memmap(
    "../data/large-data/greek_sent_embeddings.memmap",
    dtype="float32",
    mode="r",
    shape=(N, dim),  # same as you saved earlier
)

In [16]:
emb_read.shape

(1001301, 768)

In [17]:
# filter out the GNT rows:
# 1) Build the mask (True = keep row)
#mask = ~grouped["sentence_id"].str.startswith("lagt_tlg0031")

# 2) Apply to dataframe
#grouped_filtered = grouped[mask].reset_index(drop=True)

# 3) Apply the same mask to the memmap embeddings
#emb_read = emb_read[mask.values]

In [18]:
# dimension = embedding size
d = emb_read.shape[1]

# build FAISS index (cosine similarity since we L2-normalized)
index = faiss.IndexFlatIP(d)
index.add(emb_read)

# save index
faiss.write_index(index, "../data/large-data/grouped_emb.index")

In [19]:
sentences_annotated_merged = pd.read_parquet("../data/sentences_annotated_merged_emb.parquet")

In [20]:
sentences_annotated_merged["is_polemical"] = ~sentences_annotated_merged["unpolemical"].astype(bool)

In [21]:
sentences_annotated_merged["is_polemical"].value_counts()

is_polemical
False    2000
True     1269
Name: count, dtype: int64

In [83]:
# create faiss index for annotated sentences (formerly only from GNT...

# Convert list of numpy arrays to a 2D float32 array
verse_embs = np.stack(sentences_annotated_merged["sent_emb_agbert"].values).astype("float32")

# Dimension of embeddings
d = verse_embs.shape[1]

# Build FAISS index (cosine similarity if embeddings are L2-normalized)
index_nt = faiss.IndexFlatIP(d)
index_nt.add(verse_embs)

# Save to disk
faiss.write_index(index_nt, "../data/large-data/nt_verses_emb.index")

Indexed 1643 NT verses with dim=768


In [48]:
# --- Apply classifier to all filtered embeddings ---
prob_threshold = 0.9  # set threshold for "polemical"

# Convert memmap embeddings into a proper numpy array if needed
# (faiss.IndexFlatIP expects float32, so we already stored that)
X_grouped = np.array(emb_read, dtype="float32")

# Predict probabilities (class 1 = polemical)
proba = best_clf.predict_proba(X_grouped)[:, 1]

# Apply threshold
y_pred = (proba >= prob_threshold).astype(int)

# Attach results back to your dataframe
grouped["prob_polemical"] = proba
grouped["pred_polemical"] = y_pred

print(grouped["pred_polemical"].value_counts())

pred_polemical
0    909492
1     91809
Name: count, dtype: int64


In [49]:
sum(grouped["pred_polemical"]) / len(grouped)

0.0916897116850977

In [50]:
sentences_annotated_polemical = sentences_annotated_merged[sentences_annotated_merged["is_polemical"] == 1].copy()
grouped_polemical = grouped[grouped["pred_polemical"] == 1].copy()

In [51]:
# K x D verse matrix (in verses_polemical row order)
V = np.stack(sentences_annotated_polemical["sent_emb_agbert"].values).astype("float32")   # (K, D)
K, D = V.shape

# polemical sentences on grouped side
mask = (grouped["pred_polemical"] == 1).values
grouped_polemical = grouped[mask].reset_index(drop=True)
q_idx = np.nonzero(mask)[0]                # back-ref to full grouped
Q = emb_read[q_idx]                        # (M, D), already L2-normalized

# dense cosine (dot) similarities; stays ~0.48 GB for 188k x 643
S = Q @ V.T                                # (M, K) float32

# keep only a tiny pointer in the dataframe
grouped_polemical["sim_row"] = np.arange(S.shape[0], dtype=np.int64)

In [52]:
r = grouped_polemical.iloc[0]["sim_row"]   # pick the first polemical sentence
sims = S[r]                                # similarities to *all* verses
print(sims.shape)                          # should be (len(verses_polemical),)
print(sims[:10])

(1269,)
[0.84107393 0.8514528  0.83898324 0.820211   0.84337515 0.8509512
 0.83476293 0.83948594 0.8605566  0.8420774 ]


In [53]:
# categories vector (length = n_verses_polemical)
cats = sentences_annotated_polemical["primary_category"].to_numpy()

# unique categories you care about
unique_cats = np.unique(cats)

# output: DataFrame with avg similarity per category
avg_sims = []
for r in range(S.shape[0]):   # loop over grouped_polemical sentences
    sims = S[r]  # similarity scores to all verses
    row_result = {"row": r}
    for c in unique_cats:
        mask = (cats == c)
        if mask.any():
            topk = 5
            sims_sorted = np.sort(sims[mask])[-topk:]
            row_result[f"cat_{c}_top{topk}_avg"] = sims_sorted.mean()
            #row_result[f"cat_{c}_median_sim"] = np.median(sims[mask])
        else:
            row_result[f"cat_{c}_avg_sim"] = np.nan
    avg_sims.append(row_result)

avg_sims_df = pd.DataFrame(avg_sims)

In [54]:
# 1) Get the column with the maximum value per row
cat_cols = [c for c in avg_sims_df.columns if c.startswith("cat_")]
avg_sims_df["best_category"] = avg_sims_df[cat_cols].idxmax(axis=1)
avg_sims_df["best_score"] = avg_sims_df[cat_cols].max(axis=1)

In [55]:
avg_sims_df

,row,cat_1_top5_avg,cat_2_top5_avg,cat_3_top5_avg,cat_4_top5_avg,cat_5_top5_avg,cat_6_top5_avg,best_category,best_score
0,0,0.881476,0.877600,0.877606,0.883651,0.891564,0.868506,cat_5_top5_avg,0.891564
1,1,0.884621,0.881736,0.879319,0.887714,0.889936,0.875543,cat_5_top5_avg,0.889936
2,2,0.882730,0.866127,0.870962,0.871994,0.882648,0.861038,cat_1_top5_avg,0.882730
3,3,0.914598,0.918732,0.912926,0.916691,0.924578,0.909077,cat_5_top5_avg,0.924578
4,4,0.905913,0.899617,0.899574,0.900133,0.906002,0.893005,cat_5_top5_avg,0.906002
...,...,...,...,...,...,...,...,...,...
91804,91804,0.932918,0.937639,0.929683,0.929457,0.941463,0.929739,cat_5_top5_avg,0.941463
91805,91805,0.876382,0.873896,0.872153,0.874686,0.879943,0.868155,cat_5_top5_avg,0.879943
91806,91806,0.928696,0.923447,0.923429,0.926331,0.927573,0.919954,cat_1_top5_avg,0.928696
91807,91807,0.840669,0.834195,0.827286,0.835924,0.851982,0.840732,cat_5_top5_avg,0.851982


In [56]:
avg_sims_df["best_cat_num"] = (
    avg_sims_df["best_category"].str.extract(r"cat_(\d+)").astype(int)
)

In [57]:
grouped_polemical["polemical_cat_pred"] = avg_sims_df["best_cat_num"]
grouped_polemical["polemical_cat_sim_score"] = avg_sims_df["best_score"]

In [58]:
grouped_polemical.groupby("polemical_cat_pred").size()

polemical_cat_pred
1    26782
2     8304
3     3245
4     5380
5    46441
6     1657
dtype: int64

In [59]:
grouped_polemical.to_parquet("../data/large-data/grouped_polemical.parquet")